# Квантизация LLM с помощью LLM Compressor

Практический блокнот для сравнения нескольких вариантов точности и квантизации одной LLM: **базовый BF16**, **FP8**, **NVFP4** и **INT4 (W4A16 + GPTQ)**.

Каждый вариант проходит одинаковую оценку, а его метрики сохраняются только в `RESULTS` в оперативной памяти до итогового сравнения.


## Теоретическая часть

### BF16 — базовый вариант, а не квантизация

`BF16` использует 16 бит на значение с плавающей точкой и в этом блокноте служит исходной точкой сравнения. Технически это **формат с плавающей точкой пониженной точности**, а не схема квантизации LLM Compressor.

В текстовой конфигурации `Qwen/Qwen3.5-2B` уже задано `dtype=bfloat16`, поэтому мы не называем этот этап «квантизацией в BF16».


### Посттренировочная квантизация (PTQ)

В блокноте используется **Post-Training Quantization (PTQ)** — посттренировочная квантизация: исходная модель уже обучена, а низкоразрядное представление создаётся после обучения.

Некоторые схемы могут применяться без дополнительных данных, а другие используют калибровочный датасет для определения параметров квантизации или более точного восстановления весов.

### FP8

`FP8_DYNAMIC` в LLM Compressor — схема квантизации W8A8 с плавающей точкой: веса хранятся в 8-битном формате FP8, а активации квантуются динамически во время инференса.

Для варианта RTN калибровочный датасет не требуется. Теоретически веса занимают примерно в два раза меньше места, чем в BF16, без перехода к экстремальной 4-битной точности.

### NVFP4

`NVFP4` — 4-битная схема с плавающей точкой для архитектуры Blackwell. Она квантует веса и активации и использует двухуровневую схему масштабирования: значения FP4 E2M1, локальные блоки по 16 значений, коэффициенты масштабирования блоков в FP8 и глобальный коэффициент масштабирования.

Для вычисления глобальных коэффициентов масштабирования активаций требуется калибровочный датасет.

### INT4 W4A16

`W4A16` квантует веса до 4-битного целочисленного представления, а активации оставляет 16-битными. В блокноте для W4A16 используется `GPTQModifier`, то есть веса оптимизируются с помощью калибровочных данных, а не просто округляются методом RTN.

Это позволяет сравнить две принципиально разные 4-битные стратегии: **NVFP4 W4A4** и **INT4 W4A16**.

### Почему варианты должны начинаться с одной и той же BF16 модели

FP8, NVFP4 и INT4 нельзя применять последовательно к уже квантованной модели. Каждый эксперимент должен начинаться с новой загрузки одной и той же исходной модели BF16 одной и той же ревизии.

Иначе второй этап квантизации измерял бы уже не эффект выбранной схемы относительно оригинала, а эффект повторной квантизации предыдущего артефакта.

### Единая оценка

Каждый вариант точности или квантизации оценивается на одной и той же подвыборке валидационной части SST-2.

Используются:

- точность по сгенерированным ответам;
- доля корректных выходных ответов;
- точность при принудительном выборе;
- Precision / Recall / F1 по классам;
- Macro F1;
- матрица ошибок;
- время генерации.


### Накопление результатов в памяти


После завершения каждого эксперимента его метрики записываются в `RESULTS`. Данные существуют только в памяти текущего ядра и используются в конце блокнота для общей таблицы и графиков.


### Порядок эксперимента


Каждый эксперимент заканчивается одинаково:

1. оценкой модели;
2. записью метрик в `RESULTS`;
3. освобождением модели и очисткой памяти GPU.

Никакой отдельный файл с результатами бенчмарка не создаётся.


### Особенность Qwen3.5

Для текстовой оценки используется `AutoModelForCausalLM`, который загружает текстовую часть Qwen3.5. Проекции линейного внимания исключаются из рецептов квантизации, поскольку объединённые проекции Gated DeltaNet имеют отдельные ограничения совместимости в актуальных рецептах для Qwen3.5.

### Теоретические источники

- [LLM Compressor — Compression Schemes](https://docs.vllm.ai/projects/llm-compressor/en/stable/guides/compression_schemes/)
- [LLM Compressor — Choosing the right compression scheme](https://docs.vllm.ai/projects/llm-compressor/en/stable/steps/choosing-scheme/)
- [LLM Compressor — INT4 Weight Quantization](https://docs.vllm.ai/projects/llm-compressor/en/latest/examples/quantization_w4a16/)
- [LLM Compressor — FP4 Quantization with NVFP4](https://docs.vllm.ai/projects/llm-compressor/en/latest/examples/quantization_w4a4_fp4/)
- [LLM Compressor — oneshot](https://docs.vllm.ai/projects/llm-compressor/en/latest/guides/entrypoints/oneshot/)
- [Transformers — Qwen3.5](https://huggingface.co/docs/transformers/model_doc/qwen3_5)
- [Qwen3.5-2B config](https://huggingface.co/Qwen/Qwen3.5-2B/blob/main/config.json)

## Практическая часть

### 1. Импорты и проверка среды

Зависимости устанавливаются в Docker-образе, а не внутри блокнота. Блокнот использует PyTorch, Transformers, Datasets, TorchMetrics, Plotly и LLM Compressor.

In [1]:
import gc
import sys
from importlib.metadata import version as package_version
from pathlib import Path
from time import perf_counter

import plotly.graph_objects as go
import torch
from datasets import load_dataset
from huggingface_hub import HfApi
from torchmetrics import MetricCollection
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassConfusionMatrix,
    MulticlassF1Score,
    MulticlassPrecision,
    MulticlassRecall,
)
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

from compressed_tensors.offload import dispatch_model
from llmcompressor import oneshot
from llmcompressor.modifiers.gptq import GPTQModifier
from llmcompressor.modifiers.quantization import QuantizationModifier

print(f"Python:              {sys.version.split()[0]}")
print(f"PyTorch:             {torch.__version__}")
print(f"Transformers:        {package_version('transformers')}")
print(f"Datasets:            {package_version('datasets')}")
print(f"TorchMetrics:        {package_version('torchmetrics')}")
print(f"LLM Compressor:      {package_version('llmcompressor')}")
print(f"compressed-tensors:  {package_version('compressed-tensors')}")
print(f"Hugging Face Hub:    {package_version('huggingface_hub')}")

assert torch.cuda.is_available(), "This quantization notebook requires CUDA."

DEVICE = torch.device("cuda")

set_seed(42)
torch.set_float32_matmul_precision("high")


Python:              3.10.12
PyTorch:             2.13.0+cu130
Transformers:        5.14.1
Datasets:            5.0.1
TorchMetrics:        1.9.0
LLM Compressor:      0.13.0
compressed-tensors:  0.18.0
Hugging Face Hub:    1.28.0


### 2. Конфигурация

`RUN_MODE='smoke'` предназначен для быстрой проверки всего конвейера. `RUN_MODE='full'` увеличивает объём калибровки и оценки.


In [2]:
SEED = 42

MODEL_ID = "Qwen/Qwen3.5-2B"
MODEL_REVISION = None

DATASET_ID = "stanfordnlp/sst2"
DATASET_REVISION = None

TEXT_COLUMN = "sentence"
LABEL_COLUMN = "label"

RUN_MODE = "smoke"
assert RUN_MODE in {"smoke", "full"}

MAX_PROMPT_LENGTH = 256
MAX_NEW_TOKENS = 4

NUM_CALIBRATION_SAMPLES = (
    64 if RUN_MODE == "smoke" else 512
)

MAX_EVAL_SAMPLES = (
    256 if RUN_MODE == "smoke" else None
)

GENERATION_BATCH_SIZE = 16
FORCED_CHOICE_BATCH_SIZE = 32

QUANTIZATION_TARGETS = "Linear"

COMMON_IGNORE = [
    "lm_head",
    "re:.*linear_attn.*",
]

VARIANTS = (
    "BF16",
    "FP8",
    "NVFP4",
    "INT4",
)

RESULTS = {}

set_seed(SEED)

print(f"Run mode:           {RUN_MODE}")
print(f"Calibration:        {NUM_CALIBRATION_SAMPLES}")


Run mode:           smoke
Calibration:        64


### 3. Загрузка SST-2

Обучающая часть используется только для калибровки. Валидационная часть используется только для сравнения BF16, FP8, NVFP4 и INT4.

In [3]:
raw_dataset = load_dataset(
    DATASET_ID,
    revision=DATASET_REVISION,
)

assert "train" in raw_dataset
assert "validation" in raw_dataset

resolved_dataset_revision = DATASET_REVISION

if not resolved_dataset_revision:
    try:
        resolved_dataset_revision = HfApi().dataset_info(
            DATASET_ID
        ).sha
    except Exception:
        resolved_dataset_revision = "unavailable"

print(raw_dataset)
print(f"Dataset revision: {resolved_dataset_revision}")
print(f"Обучающих примеров:   {len(raw_dataset['train']):,}")
print(f"Валидационных примеров: {len(raw_dataset['validation']):,}")

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})
Dataset revision: 8d51e7e4887a4caaa95b3fbebbf53c0490b58bbb
Обучающих примеров:   67,349
Валидационных примеров: 872


### 4. Токенизатор и формат задачи

Все эксперименты используют один токенизатор, один шаблон чата и один формат выходного ответа. Это исключает расхождения в промптах между вариантами точности.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

VISIBLE_INSTRUCTION = (
    "Classify the sentiment of this movie review as positive or negative. "
    "Answer with exactly one word: negative or positive. "
    "Do not provide explanations or reasoning."
)


def build_user_text(text):
    return (
        f"{VISIBLE_INSTRUCTION}\n"
        f"Review: {text.strip()}"
    )


def render_prompt(text):
    messages = [
        {
            "role": "user",
            "content": build_user_text(text),
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


print(
    render_prompt(
        raw_dataset["validation"][0][TEXT_COLUMN]
    )
)


<|im_start|>user
Classify the sentiment of this movie review as positive or negative. Answer with exactly one word: negative or positive. Do not provide explanations or reasoning.
Review: it 's a charming and often affecting journey .<|im_end|>
<|im_start|>assistant
<think>

</think>




### 5. Общие функции загрузки модели

Каждый вариант квантизации начинается с новой модели, загруженной через `fresh_model()`. Это гарантирует, что FP8, NVFP4 и INT4 сравниваются непосредственно с одной и той же ревизией BF16.

In [5]:
def fresh_model():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        dtype=torch.bfloat16,
    )

    model = model.to(DEVICE)
    model.eval()

    return model


def release_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()


probe_model = fresh_model()

resolved_model_revision = (
    MODEL_REVISION
    or getattr(probe_model.config, "_commit_hash", None)
)

if not resolved_model_revision:
    try:
        resolved_model_revision = HfApi().model_info(
            MODEL_ID
        ).sha
    except Exception:
        resolved_model_revision = "unavailable"

parameter_count = sum(
    parameter.numel()
    for parameter in probe_model.parameters()
)

print(f"Loaded class:   {type(probe_model).__name__}")
print(f"Parameters:     {parameter_count:,}")
print(f"Model revision: {resolved_model_revision}")

del probe_model
gc.collect()
torch.cuda.empty_cache()


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loaded class:   Qwen3_5ForCausalLM
Parameters:     1,881,825,088
Model revision: 15852e8c16360a2fea060d615a32b45270f8a8fc


### 6. TorchMetrics и оценка

Стандартные метрики классификации объединены в `TorchMetrics.MetricCollection`. Оценка по генерации и оценка с принудительным выбором используются для всех вариантов без изменений.

#### Оценка по генерации

Модель самостоятельно генерирует продолжение. Ответ считается корректным, только если начинается с `negative` или `positive`.

#### Оценка с принудительным выбором

Для однотокенных меток сравниваются логиты следующего токена для двух допустимых ответов.

$\hat y = \arg\max_{y \in \{\text{negative},\text{positive}\}} z_y$

In [6]:
CLASS_LABELS = ("negative", "positive")
INVALID_LABEL_ID = len(CLASS_LABELS)
NUM_EVAL_CLASSES = INVALID_LABEL_ID + 1

EVAL_METRICS = MetricCollection(
    {
        "accuracy": MulticlassAccuracy(NUM_EVAL_CLASSES, average="micro"),
        "precision": MulticlassPrecision(
            NUM_EVAL_CLASSES,
            average=None,
            zero_division=0,
        ),
        "recall": MulticlassRecall(
            NUM_EVAL_CLASSES,
            average=None,
            zero_division=0,
        ),
        "f1": MulticlassF1Score(
            NUM_EVAL_CLASSES,
            average=None,
            zero_division=0,
        ),
        "confusion_matrix": MulticlassConfusionMatrix(NUM_EVAL_CLASSES),
    }
)


def resolve_label_token_ids():
    for prefix in ("", " "):
        token_ids = [
            tokenizer(prefix + label, add_special_tokens=False)["input_ids"]
            for label in CLASS_LABELS
        ]
        if all(len(ids) == 1 for ids in token_ids):
            return torch.tensor([ids[0] for ids in token_ids], dtype=torch.long)

    raise ValueError(
        "Forced-choice evaluation requires single-token sentiment labels."
    )


LABEL_TOKEN_IDS = resolve_label_token_ids()

EVAL_SPLIT = raw_dataset["validation"]
if MAX_EVAL_SAMPLES is not None:
    EVAL_SPLIT = EVAL_SPLIT.select(
        range(min(MAX_EVAL_SAMPLES, len(EVAL_SPLIT)))
    )


def iter_batches(dataset, batch_size):
    for start in range(0, len(dataset), batch_size):
        yield dataset[start:start + batch_size]


def tokenize_prompts(texts, model_device):
    return tokenizer(
        [render_prompt(text) for text in texts],
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_PROMPT_LENGTH,
        padding=True,
        return_tensors="pt",
    ).to(model_device)


def normalize_prediction(text):
    text = text.strip().lower()
    for index, label in enumerate(CLASS_LABELS):
        if text.startswith(label):
            return index
    return INVALID_LABEL_ID


def classification_metrics(predictions, references):
    predictions = torch.tensor(predictions, dtype=torch.long)
    references = torch.tensor(references, dtype=torch.long)
    values = EVAL_METRICS.clone()(predictions, references)

    precision = values["precision"][: len(CLASS_LABELS)]
    recall = values["recall"][: len(CLASS_LABELS)]
    f1 = values["f1"][: len(CLASS_LABELS)]

    return {
        "accuracy": values["accuracy"].item(),
        "macro_f1": f1.mean().item(),
        "per_class": {
            label: {
                "precision": precision[index].item(),
                "recall": recall[index].item(),
                "f1": f1[index].item(),
            }
            for index, label in enumerate(CLASS_LABELS)
        },
        "confusion_matrix": (
            values["confusion_matrix"][: len(CLASS_LABELS)]
            .to(torch.int64)
            .tolist()
        ),
        "total": len(references),
    }


def print_classification_summary(title, metrics):
    print(f"\n{title}\n{'-' * len(title)}")
    print(f"Accuracy:          {metrics['accuracy']:.2%}")
    print(f"Macro F1:          {metrics['macro_f1']:.4f}")

    for label in CLASS_LABELS:
        values = metrics["per_class"][label]
        print(
            f"{label:8s} "
            f"precision={values['precision']:.4f} "
            f"recall={values['recall']:.4f} "
            f"f1={values['f1']:.4f}"
        )

    if "valid_output_rate" in metrics:
        print(f"Доля корректных ответов: {metrics['valid_output_rate']:.2%}")
        print(
            "Время генерации:        "
            f"{metrics['milliseconds_per_sample']:.2f} ms/sample"
        )


def generate(model, inputs):
    return model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )


def warmup_generation(model):
    model_device = next(model.parameters()).device
    sample = EVAL_SPLIT[: min(4, len(EVAL_SPLIT))]

    with torch.inference_mode():
        generate(
            model,
            tokenize_prompts(sample[TEXT_COLUMN], model_device),
        )


def evaluate_generation(model):
    model.eval()
    model_device = next(model.parameters()).device
    predictions = []
    references = []

    torch.cuda.synchronize()
    start_time = perf_counter()

    with torch.inference_mode():
        for batch in iter_batches(EVAL_SPLIT, GENERATION_BATCH_SIZE):
            inputs = tokenize_prompts(batch[TEXT_COLUMN], model_device)
            outputs = generate(model, inputs)

            generated_texts = tokenizer.batch_decode(
                outputs[:, inputs["input_ids"].shape[1]:],
                skip_special_tokens=True,
            )

            predictions.extend(
                normalize_prediction(text)
                for text in generated_texts
            )
            references.extend(
                int(label_id)
                for label_id in batch[LABEL_COLUMN]
            )

    torch.cuda.synchronize()
    elapsed = perf_counter() - start_time

    metrics = classification_metrics(predictions, references)
    metrics.update(
        {
            "valid_output_rate": (
                sum(p != INVALID_LABEL_ID for p in predictions)
                / len(EVAL_SPLIT)
            ),
            "milliseconds_per_sample": (
                elapsed / len(EVAL_SPLIT) * 1_000
            ),
        }
    )
    return metrics


def evaluate_forced_choice(model):
    model.eval()
    model_device = next(model.parameters()).device
    label_token_ids = LABEL_TOKEN_IDS.to(model_device)
    predictions = []
    references = []

    with torch.inference_mode():
        for batch in iter_batches(EVAL_SPLIT, FORCED_CHOICE_BATCH_SIZE):
            inputs = tokenize_prompts(batch[TEXT_COLUMN], model_device)
            label_logits = (
                model(**inputs)
                .logits[:, -1, :]
                .index_select(-1, label_token_ids)
            )

            predictions.extend(label_logits.argmax(dim=-1).tolist())
            references.extend(
                int(label_id)
                for label_id in batch[LABEL_COLUMN]
            )

    return classification_metrics(predictions, references)


def evaluate_model(model, variant_name):
    warmup_generation(model)
    generation = evaluate_generation(model)
    forced_choice = evaluate_forced_choice(model)

    print_classification_summary(
        f"{variant_name} — оценка по генерации",
        generation,
    )
    print_classification_summary(
        f"{variant_name} — принудительный выбор",
        forced_choice,
    )

    return generation, forced_choice


### 7. Калибровочный датасет

Одна калибровочная подвыборка подготавливается заранее и затем повторно используется для NVFP4 и INT4 GPTQ. `FP8_DYNAMIC` в этом блокноте использует RTN и не требует калибровочного датасета.

In [7]:
calibration_source = (
    raw_dataset["train"]
    .shuffle(seed=SEED)
    .select(
        range(
            min(
                NUM_CALIBRATION_SAMPLES,
                len(raw_dataset["train"]),
            )
        )
    )
)


def prepare_calibration_example(example):
    encoded = tokenizer(
        render_prompt(example[TEXT_COLUMN]),
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_PROMPT_LENGTH,
    )

    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
    }


calibration_dataset = calibration_source.map(
    prepare_calibration_example,
    remove_columns=calibration_source.column_names,
    desc="Подготовка калибровочных данных",
)


def calibration_data_collator(batch):
    assert len(batch) == 1

    return {
        key: torch.tensor(
            value,
            dtype=torch.long,
        ).unsqueeze(0)
        for key, value in batch[0].items()
    }


print(calibration_dataset)
print(
    f"Калибровочных примеров: "
    f"{len(calibration_dataset)}"
)

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 64
})
Калибровочных примеров: 64


### 8. Накопление результатов и опциональное сохранение модели


Метрики каждого варианта записываются только в словарь `RESULTS` в оперативной памяти. Повторный запуск ячейки обновляет соответствующий вариант.

Сохранение самой модели отделено от результатов: `save_variant()` можно вызвать вручную при необходимости. Метрики и общий бенчмарк эта функция на диск не записывает.


In [8]:
def record_result(
    variant_name,
    generation_metrics,
    forced_choice_metrics,
):
    RESULTS[variant_name] = {
        "variant": variant_name,
        "generation_accuracy": generation_metrics["accuracy"],
        "forced_choice_accuracy": forced_choice_metrics["accuracy"],
        "generation_macro_f1": generation_metrics["macro_f1"],
        "forced_choice_macro_f1": forced_choice_metrics["macro_f1"],
        "valid_output_rate": generation_metrics["valid_output_rate"],
        "generation_ms_per_sample": (
            generation_metrics["milliseconds_per_sample"]
        ),
    }


def save_variant(
    model,
    output_dir,
    *,
    save_compressed=True,
    hub_id=None,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(
        output_dir,
        save_compressed=save_compressed,
    )
    tokenizer.save_pretrained(output_dir)

    if hub_id is not None:
        api = HfApi()
        api.create_repo(
            repo_id=hub_id,
            repo_type="model",
            exist_ok=True,
        )
        api.upload_folder(
            folder_path=output_dir,
            repo_id=hub_id,
            repo_type="model",
        )


### 9. Базовый BF16

Qwen3.5-2B уже распространяется с текстовой конфигурацией BF16, поэтому этот этап не является квантизацией. Он создаёт **собственный текстовый базовый артефакт** для корректного сравнения размеров всех последующих вариантов.

#### Оценка BF16

In [9]:
bf16_model = fresh_model()

bf16_generation_metrics, bf16_forced_choice_metrics = (
    evaluate_model(
        bf16_model,
        "BF16",
    )
)

record_result(
    "BF16",
    bf16_generation_metrics,
    bf16_forced_choice_metrics,
)

del bf16_model
gc.collect()
torch.cuda.empty_cache()


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]


BF16 — оценка по генерации
--------------------------
Accuracy:          90.23%
Macro F1:          0.9023
negative precision=0.8731 recall=0.9360 f1=0.9035
positive precision=0.9344 recall=0.8702 f1=0.9012
Доля корректных ответов: 100.00%
Время генерации:        9.52 ms/sample

BF16 — принудительный выбор
---------------------------
Accuracy:          90.62%
Macro F1:          0.9062
negative precision=0.8741 recall=0.9440 f1=0.9077
positive precision=0.9421 recall=0.8702 f1=0.9048


### 10. FP8_DYNAMIC

Эксперимент FP8 начинается с новой загрузки исходной модели BF16. `FP8_DYNAMIC` квантует веса до FP8 и использует динамическую квантизацию активаций. Для варианта RTN калибровочный датасет не требуется.

#### Квантизация FP8

In [10]:
fp8_model = fresh_model()

fp8_recipe = QuantizationModifier(
    targets=QUANTIZATION_TARGETS,
    scheme="FP8_DYNAMIC",
    ignore=COMMON_IGNORE,
)

oneshot(
    model=fp8_model,
    recipe=fp8_recipe,
)

dispatch_model(fp8_model)
fp8_model.eval()

print("FP8_DYNAMIC quantization completed.")


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

2026-08-23T14:43:11.2622 | __init__ | WARNING - Disabling tokenizer parallelism due to threading conflict between FastTokenizer and Datasets. Set TOKENIZERS_PARALLELISM=false to suppress this warning.
2026-08-23T14:43:19.2878 | reset | INFO - Compression lifecycle reset
2026-08-23T14:43:19.3024 | norm_calibration_context | INFO - Found 61 offset-norm modules to convert
2026-08-23T14:43:19.3057 | from_modifiers | INFO - Creating recipe from modifiers


Applying quantization config: 100%|██████████| 96/96 [00:00<00:00, 9537.25it/s]

2026-08-23T14:43:19.3392 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-08-23T14:43:19.3397 | IndependentPipeline | INFO - Inferred `DataFreePipeline` for `QuantizationModifier`
2026-08-23T14:43:19.4888 | norm_calibration_context | INFO - Restoring 61 norm modules to offset convention
2026-08-23T14:43:19.4936 | finalize | INFO - Compression lifecycle finalized for 1 modifiers


FP8_DYNAMIC quantization completed.


#### Сравнение FP8 с BF16

In [11]:
fp8_generation_metrics, fp8_forced_choice_metrics = (
    evaluate_model(
        fp8_model,
        "FP8",
    )
)

print(
    "\nDelta vs BF16:"
    f"\nGeneration accuracy: "
    f"{fp8_generation_metrics['accuracy'] - bf16_generation_metrics['accuracy']:+.4f}"
    f"\nForced-choice accuracy: "
    f"{fp8_forced_choice_metrics['accuracy'] - bf16_forced_choice_metrics['accuracy']:+.4f}"
)

record_result(
    "FP8",
    fp8_generation_metrics,
    fp8_forced_choice_metrics,
)

del fp8_model
gc.collect()
torch.cuda.empty_cache()



FP8 — оценка по генерации
-------------------------
Accuracy:          90.23%
Macro F1:          0.9023
negative precision=0.8731 recall=0.9360 f1=0.9035
positive precision=0.9344 recall=0.8702 f1=0.9012
Доля корректных ответов: 100.00%
Время генерации:        17.89 ms/sample

FP8 — принудительный выбор
--------------------------
Accuracy:          90.62%
Macro F1:          0.9062
negative precision=0.8741 recall=0.9440 f1=0.9077
positive precision=0.9421 recall=0.8702 f1=0.9048

Delta vs BF16:
Generation accuracy: +0.0000
Forced-choice accuracy: +0.0000


### 11. NVFP4

Эксперимент NVFP4 снова начинается с новой загрузки исходной модели BF16. В отличие от `FP8_DYNAMIC`, схема W4A4 NVFP4 использует калибровочный датасет для вычисления глобальных коэффициентов масштабирования активаций.

#### Квантизация NVFP4

In [12]:
nvfp4_model = fresh_model()

nvfp4_recipe = QuantizationModifier(
    targets=QUANTIZATION_TARGETS,
    scheme="NVFP4",
    ignore=COMMON_IGNORE,
)

oneshot(
    model=nvfp4_model,
    dataset=calibration_dataset,
    recipe=nvfp4_recipe,
    max_seq_length=MAX_PROMPT_LENGTH,
    num_calibration_samples=len(calibration_dataset),
    batch_size=1,
    shuffle_calibration_samples=False,
    data_collator=calibration_data_collator,
)

dispatch_model(nvfp4_model)
nvfp4_model.eval()

print("NVFP4 quantization completed.")


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

2026-08-23T14:43:39.0083 | reset | INFO - Compression lifecycle reset
2026-08-23T14:43:39.0105 | norm_calibration_context | INFO - Found 61 offset-norm modules to convert
2026-08-23T14:43:39.0142 | from_modifiers | INFO - Creating recipe from modifiers


Applying quantization config: 100%|██████████| 96/96 [00:00<00:00, 4111.18it/s]

2026-08-23T14:43:39.0428 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-08-23T14:43:39.0432 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `QuantizationModifier`



W0823 14:43:39.070000 1911 torch/fx/_symbolic_trace.py:56] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
(25/25): Propagating: 100%|██████████| 64/64 [00:00<00:00, 772.29it/s]

2026-08-23T14:43:56.8740 | norm_calibration_context | INFO - Restoring 61 norm modules to offset convention
2026-08-23T14:43:56.8773 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
NVFP4 quantization completed.


#### Сравнение NVFP4 с BF16

In [13]:
nvfp4_generation_metrics, nvfp4_forced_choice_metrics = (
    evaluate_model(
        nvfp4_model,
        "NVFP4",
    )
)

print(
    "\nDelta vs BF16:"
    f"\nGeneration accuracy: "
    f"{nvfp4_generation_metrics['accuracy'] - bf16_generation_metrics['accuracy']:+.4f}"
    f"\nForced-choice accuracy: "
    f"{nvfp4_forced_choice_metrics['accuracy'] - bf16_forced_choice_metrics['accuracy']:+.4f}"
)

record_result(
    "NVFP4",
    nvfp4_generation_metrics,
    nvfp4_forced_choice_metrics,
)

del nvfp4_model
gc.collect()
torch.cuda.empty_cache()



NVFP4 — оценка по генерации
---------------------------
Accuracy:          89.45%
Macro F1:          0.8945
negative precision=0.8712 recall=0.9200 f1=0.8949
positive precision=0.9194 recall=0.8702 f1=0.8941
Доля корректных ответов: 100.00%
Время генерации:        29.09 ms/sample

NVFP4 — принудительный выбор
----------------------------
Accuracy:          90.23%
Macro F1:          0.9023
negative precision=0.8731 recall=0.9360 f1=0.9035
positive precision=0.9344 recall=0.8702 f1=0.9012

Delta vs BF16:
Generation accuracy: -0.0078
Forced-choice accuracy: -0.0039


### 12. INT4 W4A16 + GPTQ

Последний эксперимент использует 4-битные целочисленные веса и 16-битные активации. `GPTQModifier` использует калибровочные примеры для оптимизации весов W4A16.

#### Квантизация INT4

In [14]:
int4_model = fresh_model()

int4_recipe = GPTQModifier(
    targets=QUANTIZATION_TARGETS,
    scheme="W4A16",
    ignore=COMMON_IGNORE,
)

oneshot(
    model=int4_model,
    dataset=calibration_dataset,
    recipe=int4_recipe,
    max_seq_length=MAX_PROMPT_LENGTH,
    num_calibration_samples=len(calibration_dataset),
    batch_size=1,
    shuffle_calibration_samples=False,
    data_collator=calibration_data_collator,
)

dispatch_model(int4_model)
int4_model.eval()

print("INT4 W4A16 GPTQ quantization completed.")


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

2026-08-23T14:44:19.5963 | reset | INFO - Compression lifecycle reset
2026-08-23T14:44:19.5986 | norm_calibration_context | INFO - Found 61 offset-norm modules to convert
2026-08-23T14:44:19.6024 | from_modifiers | INFO - Creating recipe from modifiers


Applying quantization config: 100%|██████████| 96/96 [00:00<00:00, 8361.26it/s]

2026-08-23T14:44:19.6205 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-08-23T14:44:19.6210 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`



(2/25): Calibrating: 100%|██████████| 64/64 [00:00<00:00, 158.96it/s]


...

(25/25): Calibrating: 100%|██████████| 64/64 [00:00<00:00, 465.57it/s]

2026-08-23T14:46:43.7178 | compress_module_list | INFO - Quantizing model.layers.23.self_attn.q_proj using 64 samples


2026-08-23T14:46:44.6444 | GPTQ | METRIC - time 0.93s
2026-08-23T14:46:44.6451 | GPTQ | METRIC - error 64.09
2026-08-23T14:46:44.6461 | GPTQ | METRIC - Accelerator 0 | usage: 29.29% | total memory: 34.2 Gb
2026-08-23T14:46:44.6469 | compress_module_list | INFO - Quantizing model.layers.23.self_attn.k_proj using 64 samples
2026-08-23T14:46:45.6077 | GPTQ | METRIC - time 0.96s
2026-08-23T14:46:45.6082 | GPTQ | METRIC - error 8.13
2026-08-23T14:46:45.6093 | GPTQ | METRIC - Accelerator 0 | usage: 29.29% | total memory: 34.2 Gb
2026-08-23T14:46:45.6100 | compress_module_list | INFO - Quantizing model.layers.23.self_attn.v_proj using 64 samples
2026-08-23T14:46:46.5707 | GPTQ | METRIC - time 0.96s
2026-08-23T14:46:46.5713 | GPTQ | METRIC - error 29.44
2026-08-23T14:46:46.5722 | GPTQ | METRIC - Accelerator 0 | usage: 29.29% | total memory: 34.2 Gb
2026-08-23T14:46:46.5728 | compress_module_list | INFO - Quantizing model.layers.23.self_attn.o_proj using 64 samples
2026-08-23T14:46:47.6530 | GP

(25/25): Propagating: 100%|██████████| 64/64 [00:00<00:00, 550.39it/s]

2026-08-23T14:46:52.3418 | norm_calibration_context | INFO - Restoring 61 norm modules to offset convention
2026-08-23T14:46:52.3448 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
INT4 W4A16 GPTQ quantization completed.


#### Сравнение INT4 с BF16

In [15]:
int4_generation_metrics, int4_forced_choice_metrics = (
    evaluate_model(
        int4_model,
        "INT4",
    )
)

print(
    "\nDelta vs BF16:"
    f"\nGeneration accuracy: "
    f"{int4_generation_metrics['accuracy'] - bf16_generation_metrics['accuracy']:+.4f}"
    f"\nForced-choice accuracy: "
    f"{int4_forced_choice_metrics['accuracy'] - bf16_forced_choice_metrics['accuracy']:+.4f}"
)

record_result(
    "INT4",
    int4_generation_metrics,
    int4_forced_choice_metrics,
)

del int4_model
gc.collect()
torch.cuda.empty_cache()



INT4 — оценка по генерации
--------------------------
Accuracy:          90.62%
Macro F1:          0.9062
negative precision=0.8797 recall=0.9360 f1=0.9070
positive precision=0.9350 recall=0.8779 f1=0.9055
Доля корректных ответов: 100.00%
Время генерации:        13.73 ms/sample

INT4 — принудительный выбор
---------------------------
Accuracy:          89.84%
Macro F1:          0.8984
negative precision=0.8667 recall=0.9360 f1=0.9000
positive precision=0.9339 recall=0.8626 f1=0.8968

Delta vs BF16:
Generation accuracy: +0.0039
Forced-choice accuracy: -0.0078


### 13. Итоговое сравнение всех вариантов

`RESULTS` уже содержит строки в порядке выполнения экспериментов. На этом этапе накопленные результаты выводятся вместе и используются для итоговых графиков.

In [16]:
missing_variants = [
    variant
    for variant in VARIANTS
    if variant not in RESULTS
]

if missing_variants:
    raise RuntimeError(
        "Нет результатов для вариантов: "
        + ", ".join(missing_variants)
    )

results = [
    RESULTS[variant]
    for variant in VARIANTS
]

print(
    f"{'Variant':10s}"
    f"{'Gen Acc':>10s}"
    f"{'FC Acc':>10s}"
    f"{'Gen F1':>10s}"
    f"{'FC F1':>10s}"
    f"{'Valid':>10s}"
    f"{'ms/sample':>12s}"
)

print("-" * 72)

for result in results:
    print(
        f"{result['variant']:10s}"
        f"{result['generation_accuracy']:10.4f}"
        f"{result['forced_choice_accuracy']:10.4f}"
        f"{result['generation_macro_f1']:10.4f}"
        f"{result['forced_choice_macro_f1']:10.4f}"
        f"{result['valid_output_rate']:10.4f}"
        f"{result['generation_ms_per_sample']:12.2f}"
    )


Variant      Gen Acc    FC Acc    Gen F1     FC F1     Valid   ms/sample
------------------------------------------------------------------------
BF16          0.9023    0.9062    0.9023    0.9062    1.0000        9.52
FP8           0.9023    0.9062    0.9023    0.9062    1.0000       17.89
NVFP4         0.8945    0.9023    0.8945    0.9023    1.0000       29.09
INT4          0.9062    0.8984    0.9062    0.8984    1.0000       13.73


#### График качества

In [17]:
quality_fig = go.Figure()

quality_fig.add_trace(
    go.Bar(
        name="Точность по генерации",
        x=[
            result["variant"]
            for result in results
        ],
        y=[
            result["generation_accuracy"]
            for result in results
        ],
    )
)

quality_fig.add_trace(
    go.Bar(
        name="Точность с принудительным выбором",
        x=[
            result["variant"]
            for result in results
        ],
        y=[
            result["forced_choice_accuracy"]
            for result in results
        ],
    )
)

quality_fig.update_layout(
    title="Сохранение качества",
    barmode="group",
    yaxis_title="Точность",
    yaxis_range=[0, 1],
    height=430,
)

quality_fig.show()

#### График времени генерации

In [18]:
latency_fig = go.Figure(
    data=[
        go.Bar(
            x=[
                result["variant"]
                for result in results
            ],
            y=[
                result["generation_ms_per_sample"]
                for result in results
            ],
        )
    ]
)

latency_fig.update_layout(
    title="Локальное время генерации",
    yaxis_title="ms/sample",
    height=430,
)

latency_fig.show()

## Результаты

* **BF16** показал лучший общий баланс между качеством и скоростью. `Gen Acc = 0.9023`, `FC Acc = 0.9062`, а время генерации составило всего **9.52 ms/sample** - минимальное среди всех вариантов.
* **FP8** полностью сохранил качество BF16: значения Accuracy и Macro F1 совпали с исходной моделью. Однако время генерации выросло до **17.89 ms/sample**, поэтому в данном эксперименте FP8 не дал преимущества по скорости.
* **NVFP4** показал небольшое снижение качества: `Gen Acc = 0.8945`, `FC Acc = 0.9023`. Одновременно он оказался самым медленным вариантом - **29.09 ms/sample**. В рамках этого запуска NVFP4 не продемонстрировал преимущества ни по качеству, ни по скорости.
* **INT4 W4A16 + GPTQ** показал наивысшую точность при генеративной оценке: `Gen Acc = 0.9062`. При принудительном выборе результат оказался немного ниже - `FC Acc = 0.8984`. Время генерации составило **13.73 ms/sample** - быстрее FP8 и NVFP4, но медленнее BF16.
* Для всех вариантов `Valid = 1.0000`: квантизация не привела к появлению ответов, которые не соответствуют ожидаемому формату `negative` / `positive`.

Различия в качестве между вариантами невелики. В `smoke`-режиме оценка проводится максимум на **256 примерах**, поэтому изменение Accuracy примерно на `0.0039` соответствует всего одному примеру, а `0.0078` - двум. Эти результаты подходят для проверки работоспособности методов и выявления грубой деградации, но недостаточны для статистически уверенного ранжирования вариантов по качеству.

В данном запуске ни один из вариантов квантизации не ускорил генерацию относительно BF16. Поэтому по измеренным здесь характеристикам **BF16 остаётся наиболее сбалансированным вариантом**, FP8 интересен практически полным сохранением исходного качества, а INT4 - хорошим соотношением качества и скорости среди протестированных квантизованных вариантов. Для окончательного сравнения схем следует повторить эксперимент на полной валидационной выборке и в целевой среде инференса.

## Источники

- [LLM Compressor documentation](https://docs.vllm.ai/projects/llm-compressor/)
- [Compression Schemes](https://docs.vllm.ai/projects/llm-compressor/en/stable/guides/compression_schemes/)
- [Choosing the right compression scheme](https://docs.vllm.ai/projects/llm-compressor/en/stable/steps/choosing-scheme/)
- [INT4 Weight Quantization](https://docs.vllm.ai/projects/llm-compressor/en/latest/examples/quantization_w4a16/)
- [FP4 Quantization with NVFP4](https://docs.vllm.ai/projects/llm-compressor/en/latest/examples/quantization_w4a4_fp4/)
- [oneshot entrypoint](https://docs.vllm.ai/projects/llm-compressor/en/latest/guides/entrypoints/oneshot/)
- [Qwen3.5 — Transformers](https://huggingface.co/docs/transformers/model_doc/qwen3_5)
- [Qwen3.5-2B config](https://huggingface.co/Qwen/Qwen3.5-2B/blob/main/config.json)
- [SST-2](https://huggingface.co/datasets/stanfordnlp/sst2)